Week 14 · Day 5 — Fine-Tuning GPT-2 for Domain Text Generation
Why this matters

GPT-2 (a decoder-only transformer) can generate fluent text, but in its raw form it knows nothing about your domain (e.g., legal, medical, finance, gaming). Fine-tuning lets it speak in your domain’s voice — powerful for chatbots, assistants, or content generation.

Theory Essentials

GPT-2: trained for causal LM (predict next word).

Fine-tuning goal: adapt to a dataset (e.g., movie reviews → generate reviews).

Tokenizer: GPT-2 uses byte-pair encoding (BPE).

Trainer: Hugging Face Trainer handles batching + training loop.

Loss: causal LM cross-entropy (predict next token).

In [7]:
# --- Universal, CPU-fast, old/new transformers compatible ---
import inspect, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)

# ===== Speed settings =====
model_name = "distilgpt2"   # fastest; switch to "distilgpt2" if you want a real small model
MAX_LEN = 64
N_TRAIN = 1000
N_EVAL  = 200
BATCH   = 4

# ===== Tokenizer =====
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ===== Data =====
raw = load_dataset("imdb").remove_columns(["label"])

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

tok = raw.map(tokenize, batched=True)
tok.set_format(type="torch", columns=["input_ids", "attention_mask"])
train_ds = tok["train"].shuffle(seed=42).select(range(N_TRAIN))
eval_ds  = tok["test"].shuffle(seed=42).select(range(N_EVAL))

# ===== Model =====
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# Labels = inputs for causal LM
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ===== Build TrainingArguments with compatibility =====
ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
kwargs = {}

def add(name, value):
    if name in ta_params:
        kwargs[name] = value

add("output_dir", "./gpt2-imdb-fast")
add("overwrite_output_dir", True)
add("num_train_epochs", 1)
# per_device vs per_gpu (older versions)
if "per_device_train_batch_size" in ta_params:
    add("per_device_train_batch_size", BATCH)
    add("per_device_eval_batch_size", BATCH)
else:
    add("per_gpu_train_batch_size", BATCH)
    add("per_gpu_eval_batch_size", BATCH)
add("logging_steps", 50)
add("no_cuda", True)
# (Optional) some old versions require logging_dir to silence warnings
add("logging_dir", "./logs")

# DO NOT add evaluation_strategy/save_strategy/report_to/bf16 to stay compatible
args = TrainingArguments(**kwargs)

# ===== Trainer =====
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,           # kept here; no eval during train since no strategy set
    data_collator=collator
)

trainer.train()

# ===== Quick inference =====
prompt = "The movie was"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_length=40, do_sample=True, top_k=50, top_p=0.95)
print("Generated text:", tokenizer.decode(out[0], skip_special_tokens=True))


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Step,Training Loss
50,4.158500
100,4.030900
150,4.089300
200,4.021900
250,4.035500


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text: The movie was a perfect fit for the film adaptation of the classic "The Adventures of Robert K. Smith. As director, Dr. Smith has so much in common with the characters, characters, and


1) Core (10–15 min)
Task: Run the training for 1 epoch, then generate text starting with “The movie was”.

this was using another model not distiled gpt2: The movie was recognizableRocket scalpScene TA kinetic Probhibit TAicooho Mollypress conservation the ESV Hancock Hancockby antibiotic conservationpress RhRocket Daniel ESV stairsJDSherScene Although the theby Rh Although Daniel

2) Practice (10–15 min)
Task: Change num_train_epochs → 2. Compare generated text before and after.

The movie was a perfect fit for the film adaptation of the classic "The Adventures of Robert K. Smith. As director, Dr. Smith has so much in common with the characters, characters, and

3) Stretch (optional, 10–15 min)
Task: Try your own mini-dataset of 5–10 short domain texts (e.g., tech blog snippets). Fine-tune for 1 epoch.

In [9]:
# --- Universal, CPU-fast, old/new transformers compatible ---
import inspect, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)

# ===== Data (YOUR MINI-DATASET) =====
from datasets import Dataset

# 1) Replace these with 5–10 short texts from your domain (tech blog, running, finance, etc.)
MY_TEXTS = [
    "In PyTorch, gradients flow backward through the computation graph after loss.backward().",
    "Learning rate schedules like cosine decay can stabilize training over long runs.",
    "Vector databases store embeddings so we can retrieve semantically similar documents.",
    "LoRA adds low-rank adapters to speed up fine-tuning large language models.",
    "Batch size and sequence length control the token budget and training speed on CPU."
]

# Optional: a couple held-out snippets for quick eval/generation
MY_EVAL = [
    "We clipped gradients to avoid exploding norms during training.",
    "Retrieval augmented generation grounds the model on external knowledge."
]

# Build small train/eval datasets
raw_train = Dataset.from_dict({"text": MY_TEXTS})
raw_eval  = Dataset.from_dict({"text": MY_EVAL})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",   # keep simple/static to match the rest of your script
        truncation=True,
        max_length=MAX_LEN
    )

tok_train = raw_train.map(tokenize, batched=True).remove_columns(["text"])
tok_eval  = raw_eval.map(tokenize, batched=True).remove_columns(["text"])

tok_train.set_format(type="torch", columns=["input_ids", "attention_mask"])
tok_eval.set_format(type="torch", columns=["input_ids", "attention_mask"])

train_ds = tok_train
eval_ds  = tok_eval

# TIP: since the dataset is tiny, increase BATCH if RAM allows (e.g., BATCH=8 or 16)


# ===== Model =====
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# Labels = inputs for causal LM
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ===== Build TrainingArguments with compatibility =====
ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
kwargs = {}

def add(name, value):
    if name in ta_params:
        kwargs[name] = value

add("output_dir", "./gpt2-imdb-fast")
add("overwrite_output_dir", True)
add("num_train_epochs", 1)
# per_device vs per_gpu (older versions)
if "per_device_train_batch_size" in ta_params:
    add("per_device_train_batch_size", BATCH)
    add("per_device_eval_batch_size", BATCH)
else:
    add("per_gpu_train_batch_size", BATCH)
    add("per_gpu_eval_batch_size", BATCH)
add("logging_steps", 50)
add("no_cuda", True)
# (Optional) some old versions require logging_dir to silence warnings
add("logging_dir", "./logs")

# DO NOT add evaluation_strategy/save_strategy/report_to/bf16 to stay compatible
args = TrainingArguments(**kwargs)

# ===== Trainer =====
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,           # kept here; no eval during train since no strategy set
    data_collator=collator
)

trainer.train()

# ===== Quick inference =====
prompt = "Learning rate"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_length=40, do_sample=True, top_k=50, top_p=0.95)
print("Generated text:", tokenizer.decode(out[0], skip_special_tokens=True))


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

c:\AI-Mastery\venv\Lib\site-packages\transformers\training_args.py:1619: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


Step,Training Loss


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text: Learning rate rise in all of the major media when she said no and it didn't matter if she's not in the room.

The last day the election, the party of choice for the


Mini-Challenge (≤40 min)

Task: Fine-tune GPT-2 on IMDB subset for 2 epochs. Generate 3 completions for prompts:

“The film was absolutely”

“I couldn’t believe how”

“This is a movie that”

Acceptance Criteria: Completions clearly resemble movie reviews, not generic text.

In [10]:
# --- Universal, CPU-fast, old/new transformers compatible ---
import inspect, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)

# ===== Speed settings =====
model_name = "distilgpt2"   # fastest; switch to "distilgpt2" if you want a real small model
MAX_LEN = 64
N_TRAIN = 1000
N_EVAL  = 200
BATCH   = 4

# ===== Tokenizer =====
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ===== Data =====
raw = load_dataset("imdb").remove_columns(["label"])

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

tok = raw.map(tokenize, batched=True)
tok.set_format(type="torch", columns=["input_ids", "attention_mask"])
train_ds = tok["train"].shuffle(seed=42).select(range(N_TRAIN))
eval_ds  = tok["test"].shuffle(seed=42).select(range(N_EVAL))

# ===== Model =====
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# Labels = inputs for causal LM
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ===== Build TrainingArguments with compatibility =====
ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
kwargs = {}

def add(name, value):
    if name in ta_params:
        kwargs[name] = value

add("output_dir", "./gpt2-imdb-fast")
add("overwrite_output_dir", True)
add("num_train_epochs", 1)
# per_device vs per_gpu (older versions)
if "per_device_train_batch_size" in ta_params:
    add("per_device_train_batch_size", BATCH)
    add("per_device_eval_batch_size", BATCH)
else:
    add("per_gpu_train_batch_size", BATCH)
    add("per_gpu_eval_batch_size", BATCH)
add("logging_steps", 50)
add("no_cuda", True)
# (Optional) some old versions require logging_dir to silence warnings
add("logging_dir", "./logs")

# DO NOT add evaluation_strategy/save_strategy/report_to/bf16 to stay compatible
args = TrainingArguments(**kwargs)

# ===== Trainer =====
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,           # kept here; no eval during train since no strategy set
    data_collator=collator
)

trainer.train()

# ===== Quick inference =====
prompt = "The film was absolutely"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_length=40, do_sample=True, top_k=50, top_p=0.95)
print("Generated text:", tokenizer.decode(out[0], skip_special_tokens=True))

prompt = "I couldn’t believe how"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_length=40, do_sample=True, top_k=50, top_p=0.95)
print("Generated text:", tokenizer.decode(out[0], skip_special_tokens=True))

prompt = "This is a movie that"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_length=40, do_sample=True, top_k=50, top_p=0.95)
print("Generated text:", tokenizer.decode(out[0], skip_special_tokens=True))


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

c:\AI-Mastery\venv\Lib\site-packages\transformers\training_args.py:1619: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


Step,Training Loss
50,4.158500
100,4.030900
150,4.089300
200,4.021900
250,4.035500


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text: The film was absolutely the perfect gift made to director Alan Moore. He shot in a typical style that, as a child, would be expected in Hollywood. Moore used the movie to tell a story he


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text: I couldn’t believe how I could write this, but it's not that hard to appreciate the emotional and character-wise nature of many of my reviews. Even after my initial praise, I
Generated text: This is a movie that you are really looking forward to. I have watched this movie from about 10 minutes. I was one of them as I watch this movie. I can imagine the characters are in


Notes / Key Takeaways

GPT-2 is trained for next-word prediction (causal LM).

Fine-tuning adapts it to specific style/content.

Tokenizer choice matters (GPT-2 uses BPE).

Even small fine-tunes noticeably change generation style.

Hugging Face Trainer makes LM fine-tuning manageable.

Reflection

Why does GPT-2 need causal (left-to-right) masking during training?

How does fine-tuning affect the “voice” of generated text?

**Why does GPT-2 need causal (left-to-right) masking during training?**
Because GPT-2 is trained as an **autoregressive language model**: it learns to predict the next token given only the tokens before it. Causal masking hides future tokens during training so the model can’t “cheat” by looking ahead. This makes training match the way the model will be used at inference (generate one token at a time, left → right).

**How does fine-tuning affect the “voice” of generated text?**
Fine-tuning nudges the model’s probability distribution toward patterns in your new dataset. Even with a small dataset, the model picks up vocabulary, tone, or style from those examples. For instance, fine-tuning on technical blog snippets makes outputs more technical and jargon-rich, while fine-tuning on movie reviews shifts the model toward opinionated, emotional language.
